In [ ]:
#TODO delete this
# %pip install rustxes ipywidgets

In [10]:
# TODO Imports
import pandas as pd
import numpy as np
from typing import Tuple, Dict
from sklearn.preprocessing import StandardScaler
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')


File for preparing the data for all the modules
1. Download the .xes data from the BPI datasets and put it in a folder that is not called clean_data

In [4]:
# Convert the XES to csv
import pm4py
import os

# dictionary to store the paths to the datasets
datasets = {
    'bpi2012': 'raw_data/BPI Challenge 2012.xes',
    'bpi2017': 'raw_data/BPI Challenge 2017.xes',
}

for dataset in datasets:
    log = pm4py.read_xes(datasets[dataset])
    log_df = pm4py.convert_to_dataframe(log)

    # Saving the file to csv in a folder called clean_data
    os.makedirs('clean_data', exist_ok=True)

    # change the name of the file
    csv_path = f"clean_data/{dataset}.csv"
    log_df.to_csv(csv_path, encoding='utf-8', index=False)


D:\01 Main\02 MASTER\0001\Y2\Q4_Prep\Q1_Q2_Thesis\gemppm_code\.venv\Lib\site-packages\pm4py\utils.py:992: UserWarning: Install the optional requirement `rustxes` to import/export files faster.
  warnings.warn("Install the optional requirement `rustxes` to import/export files faster.")
D:\01 Main\02 MASTER\0001\Y2\Q4_Prep\Q1_Q2_Thesis\gemppm_code\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
parsing log, completed traces :: 100%|██████████| 31509/31509 [00:59<00:00, 530.70it/s]


In [5]:
# TODO shortcut code to convert the xes to csv
# import pm4py
# import os




   ---------------------------------------- 0.0/5.5 MB ? eta -:--:--
   ------- -------------------------------- 1.0/5.5 MB 10.0 MB/s eta 0:00:01
   ------------- -------------------------- 1.8/5.5 MB 5.0 MB/s eta 0:00:01
   ------------------- -------------------- 2.6/5.5 MB 4.1 MB/s eta 0:00:01
   ----------------------- ---------------- 3.1/5.5 MB 4.3 MB/s eta 0:00:01
   ----------------------- ---------------- 3.1/5.5 MB 4.3 MB/s eta 0:00:01
   ----------------------- ---------------- 3.1/5.5 MB 4.3 MB/s eta 0:00:01
   ----------------------- ---------------- 3.1/5.5 MB 4.3 MB/s eta 0:00:01
   -------------------------------- ------- 4.5/5.5 MB 2.6 MB/s eta 0:00:01
   -------------------------------------- - 5.2/5.5 MB 2.9 MB/s eta 0:00:01
   ---------------------------------------- 5.5/5.5 MB 2.7 MB/s  0:00:01
   ---------------------------------------- 0.0/914.9 kB ? eta -:--:--
   ----------- ---------------------------- 262.1/914.9 kB ? eta -:--:--
   --------------------------

## Data Cleaning

Make a data dictionary to store the paths to the datasets and the essential columns to keep
Drop all the other columns

In [3]:
# TODO We only work with the clean_data folder from now on
# Making a dictionary to store the paths to the datasets
import pandas as pd
from pathlib import Path

# todo remove the data dictionary from here
datasets = {
    'bpi2012': 'clean_data/bpi2012.csv',
    'bpi2017': 'clean_data/bpi2017.csv',
}

# The columns we want to keep are - CaseID, ActivityName, Timestamp
essential_columns = ['case:concept:name', 'concept:name', 'time:timestamp']

# Keep these columns but drop the rest and store it
for dataset in datasets:
    df = pd.read_csv(datasets[dataset])
    df = df[essential_columns]
    # df.to_csv(datasets[dataset], index=False)

    # rename the columns for consistency
    df = df.rename(
        columns={"case:concept:name": "case_id", "concept:name": "activity", "time:timestamp": "timestamp"}
    )
    df.to_csv(datasets[dataset], index=False)



In [ ]:
# Split the timestamp column to date and time
for dataset in datasets:
    df = pd.read_csv(datasets[dataset])
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df['date'] = df['timestamp'].dt.date
    df['time'] = df['timestamp'].dt.time
    df.to_csv(datasets[dataset], index=False)



## Summary Statistics for the datasets

Converting the available datasets into a summay table to understand the data. The columns of the summary table:
* Event log name
* Number of cases
* Number of events
* Min/ Max/ Avg Number of activities
* Min/ Max/ Avg case length
* Min/ Max/ Avg time span (days)
* Entropy of activities - how diverse are the activities in the log


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# We have a dictionary called datasets that contains the paths to the datasets

def summarize_event_logs(dataset_paths: dict) -> pd.DataFrame:
    # Load the data from the CSV  file if a file path is provided
    summaries = []

    #Iterate through all the datasets
    for dataset_path in dataset_paths:
        df = pd.read_csv(dataset_paths[dataset_path])






print(summarize_event_logs(datasets))


## Preparing the data by keeping the essential columns only

In [ ]:
from typing import Tuple, Dict

def prepare_data(
    df: pd.DataFrame,
    test_size: float = 0.2,
    min_case_length: int = 2,
    time_unit: str = 'd'
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """

    Steps:~~
    1. Keep essential columns (case_id, activity, timestamp)
    2. Remove cases with ≤ min_case_length events (too short for prediction)
    3. Sort by case and timestamp
    4. Temporal train/test split
    5. Extract time features
    6. Normalize numerical features with z-score

    Parameters:
    -----------
    df : pd.DataFrame
        Raw event log with columns: 'case:concept:name', 'concept:name', 'time:timestamp'
    test_size : float
        Proportion for test set (default: 0.2)
    min_case_length : int
        Minimum events per case (default: 2, removes single-event cases)
    time_unit : str
        Time unit for accumulated/remaining time ('d' = days)

    Returns:
    --------
    train, test : Tuple[pd.DataFrame, pd.DataFrame]
        Preprocessed train and test sets with features
    """


    original_shape = df.shape
    df = df.loc[:, ["case:concept:name", "concept:name", "time:timestamp"]].copy()
    print(f"  Reduced from {original_shape[1]} to {df.shape[1]} columns")

    # STEP 2: Remove short cases
    # Cases with ≤ min_case_length events cannot be used for prediction
    print(f"\n[2/6] Removing cases with ≤ {min_case_length} events...")
    case_lengths = df.groupby("case:concept:name").size()
    valid_cases = case_lengths[case_lengths > min_case_length].index
    df = df[df["case:concept:name"].isin(valid_cases)]
    print(f"  Kept {len(valid_cases)} cases ({len(df)} events)")

    # STEP 3: Sort by case and timestamp
    # Critical for sequence-based feature extraction
    print("\n[3/6] Sorting by case and timestamp...")
    df = df.sort_values(by=["case:concept:name", "time:timestamp"])

    # STEP 4: Temporal train/test split
    print("\n[4/6] Performing temporal train/test split...")
    train, test = temporal_train_test_split(df, test_size=test_size)

    # STEP 5: Extract time features
    print("\n[5/6] Extracting time features...")
    ts = TimestampExtractor(
        case_features=["accumulated_time", "remaining_time"],
        event_features="all",  # Extract all temporal features
        time_unit=time_unit,
    )

    # Fit on train, transform both
    train[ts.get_feature_names_out()] = ts.fit_transform(train)
    test[ts.get_feature_names_out()] = ts.transform(test)

    print(f"  Extracted {len(ts.get_feature_names_out())} time features")

    # Drop original timestamp (no longer needed)
    train = train.drop(columns=["time:timestamp"])
    test = test.drop(columns=["time:timestamp"])

    # STEP 6: Rename columns for consistency
    train = train.rename(
        columns={"case:concept:name": "case_id", "concept:name": "activity"}
    )
    test = test.rename(
        columns={"case:concept:name": "case_id", "concept:name": "activity"}
    )

    # STEP 7: Z-score normalization
    # Normalize all numerical features to mean=0, std=1
    print("\n[6/6] Normalizing numerical features...")
    sc = StandardScaler()
    columns_to_normalize = NUMERICAL_FEATURES + ["remaining_time"]

    # Fit on train, transform both (prevents data leakage)
    train.loc[:, columns_to_normalize] = sc.fit_transform(train[columns_to_normalize])
    test.loc[:, columns_to_normalize] = sc.transform(test[columns_to_normalize])

    print(f"  Normalized {len(columns_to_normalize)} features")

    print("\n" + "="*80)
    print("PREPROCESSING COMPLETE")
    print("="*80)
    print(f"Train set: {train.shape}")
    print(f"Test set:  {test.shape}")
    print(f"Columns:   {list(train.columns)}")
    print("="*80 + "\n")

    return train, test

In [ ]:
# Temporal train/test split function
def temporal_train_test_split(
        df: pd.DataFrame,
        test_size: float = 0.2,
        case_id_col: str = 'case:concept:name',
        timestamp_col: str = 'time:timestamp'
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Split data temporally to prevent data leakage.

    Strategy:
    - Sort cases by their START time (first event timestamp)
    - First (1-test_size)% of cases → train
    - Last test_size% of cases → test

    This ensures no future cases leak into training data.

    Parameters:
    -----------
    df : pd.DataFrame
        Event log with case_id and timestamp columns
    test_size : float
        Proportion of cases for test set (default: 0.2 = 20%)
    case_id_col : str
        Column name for case identifier
    timestamp_col : str
        Column name for event timestamp

    Returns:
    --------
    train_df, test_df : Tuple[pd.DataFrame, pd.DataFrame]
    """

    # Ensure timestamp is datetime
    df[timestamp_col] = pd.to_datetime(df[timestamp_col])

    # Get the start time of each case (timestamp of first event)
    case_start_times = df.groupby(case_id_col)[timestamp_col].min().reset_index()
    case_start_times.columns = [case_id_col, 'case_start_time']

    # Sort cases by their start time
    case_start_times = case_start_times.sort_values('case_start_time')

    # Calculate split point
    n_cases = len(case_start_times)
    train_cases_count = int(n_cases * (1 - test_size))

    # Split case IDs into train and test
    train_case_ids = case_start_times.iloc[:train_cases_count][case_id_col].values
    test_case_ids = case_start_times.iloc[train_cases_count:][case_id_col].values

    # Filter dataframe by case IDs
    train_df = df[df[case_id_col].isin(train_case_ids)].copy()
    test_df = df[df[case_id_col].isin(test_case_ids)].copy()

    print(f"Temporal split completed:")
    print(f"  Train: {len(train_case_ids)} cases ({len(train_df)} events)")
    print(f"  Test:  {len(test_case_ids)} cases ({len(test_df)} events)")
    print(f"  Train date range: {train_df[timestamp_col].min()} to {train_df[timestamp_col].max()}")
    print(f"  Test date range:  {test_df[timestamp_col].min()} to {test_df[timestamp_col].max()}")

    return train_df, test_df


## Extracting the time based features from the data

In [ ]:
import pandas as pd
import numpy as np
from typing import Tuple, Dict
from sklearn.preprocessing import StandardScaler
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Time-based features that will be extracted from timestamps
NUMERICAL_FEATURES = [
    "accumulated_time",      # Time since case start
    "day_of_month",          # 1-31
    "day_of_week",           # 0-6 (Monday=0)
    "day_of_year",           # 1-365
    "hour_of_day",           # 0-23
    "min_of_hour",           # 0-59
    "month_of_year",         # 1-12
    "sec_of_min",            # 0-59
    "secs_within_day",       # Seconds since midnight
    "week_of_year",          # 1-52
]

class TimestampExtractor:
    """"Class to extract time-based features from timestamps in event logs."""
    def __init__(self, case_features: list, event_features: str = "all", time_unit: str = "d"):
        self.case_features = case_features
        self.event_features = event_features
        self.time_unit = time_unit
        self.feature_names = []

        # Conversion factors to seconds
        self.time_conversions = {
            's': 1,
            'm': 60,
            'h': 3600,
            'd': 86400
        }
        self.time_multiplier = self.time_conversions[time_unit]

    def get_feature_names_out(self) -> list:
        """Returns list of all feature names that will be created"""
        return self.feature_names

    def _extract_features(self, df: pd.DataFrame, is_train: bool) -> pd.DataFrame:
        """
        Internal method to extract all time-based features.

        Process:
        1. Sort by case and timestamp
        2. Extract case-level features (accumulated_time, remaining_time)
        3. Extract event-level features (hour, day, week, etc.)
        """
        result = pd.DataFrame(index=df.index)

        # Ensure timestamp is datetime
        df['time:timestamp'] = pd.to_datetime(df['time:timestamp'])

        # Group by case to calculate case-level features
        grouped = df.groupby('case:concept:name')

        # CASE-LEVEL FEATURES
        if 'accumulated_time' in self.case_features:
            # Time elapsed since case start (in specified time_unit)
            case_start_times = grouped['time:timestamp'].transform('min')
            accumulated_seconds = (df['time:timestamp'] - case_start_times).dt.total_seconds()
            result['accumulated_time'] = accumulated_seconds / self.time_multiplier

        if 'remaining_time' in self.case_features:
            # Time remaining until case end (in specified time_unit)
            case_end_times = grouped['time:timestamp'].transform('max')
            remaining_seconds = (case_end_times - df['time:timestamp']).dt.total_seconds()
            result['remaining_time'] = remaining_seconds / self.time_multiplier

        # EVENT-LEVEL FEATURES
        # Extract temporal patterns from timestamps
        if self.event_features == "all":
            result['day_of_month'] = df['time:timestamp'].dt.day
            result['day_of_week'] = df['time:timestamp'].dt.dayofweek  # Monday=0, Sunday=6
            result['day_of_year'] = df['time:timestamp'].dt.dayofyear
            result['hour_of_day'] = df['time:timestamp'].dt.hour
            result['min_of_hour'] = df['time:timestamp'].dt.minute
            result['month_of_year'] = df['time:timestamp'].dt.month
            result['sec_of_min'] = df['time:timestamp'].dt.second
            result['week_of_year'] = df['time:timestamp'].dt.isocalendar().week

            # Seconds since midnight (useful for detecting daily patterns)
            result['secs_within_day'] = (
                df['time:timestamp'].dt.hour * 3600 +
                df['time:timestamp'].dt.minute * 60 +
                df['time:timestamp'].dt.second
            )

        # Store feature names on first fit
        if is_train:
            self.feature_names = result.columns.tolist()

        return result


## Basic cleaning steps
1. Removing the columns that are not needed for the analysis
2. Handling missing values
3. Converting data types if necessary
4. Saving the cleaned data back to csv